<a href="https://colab.research.google.com/github/Gaurav14cs17/Bayer_Low_light_Image_Enhancement/blob/main/PTQ4SAM_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
# Phase 4 Autonomous Fix: Bypassing Python 3.12 incompatibility via isolated Python 3.9 environment
export PATH="/content/miniconda3/bin:$PATH"

echo "--- Setting up isolated Python 3.9 environment via Miniconda ---"
if [ ! -d "/content/miniconda3" ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
  bash miniconda.sh -b -p /content/miniconda3
  rm miniconda.sh
fi

# Accept Conda Terms of Service
/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
/content/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

if ! /content/miniconda3/bin/conda env list | grep -q 'ptq4sam'; then
  /content/miniconda3/bin/conda create -n ptq4sam python=3.9 -y
fi

# Use absolute paths to force execution inside the Python 3.9 environment
PY_BIN="/content/miniconda3/envs/ptq4sam/bin/python"
PIP_BIN="/content/miniconda3/envs/ptq4sam/bin/pip"
MIM_BIN="/content/miniconda3/envs/ptq4sam/bin/mim"

echo "--- Installing PyTorch, MMCV, and Dependencies ---"
$PIP_BIN install -q --upgrade pip
$PIP_BIN install -q torch==2.0.0 torchvision==0.15.1 --index-url https://download.pytorch.org/whl/cu118
$PIP_BIN install -q -U openmim
$MIM_BIN install -q "mmcv-full==1.7.2" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html
$MIM_BIN install -q "mmdet<3.0.0"

# Install remaining project dependencies including missing 'py', 'timm', 'easydict' and pin numpy to <2
$PIP_BIN install -q pycocotools opencv-python onnxruntime onnx scipy pyyaml yapf==0.40.1 py "numpy<2" timm easydict
$PIP_BIN install -q git+https://github.com/facebookresearch/segment-anything.git

echo -e "\n--- Running Evaluation: SAM-B | W6A6 | YOLOX-l ---"
# Copy the repository to local disk to avoid Google Drive's restrictions on symbolic links
cp -r /content/drive/MyDrive/PTQ4SAM-Evaluation/PTQ4SAM /content/PTQ4SAM_local
cd /content/PTQ4SAM_local

# Link the COCO dataset to the expected path
mkdir -p data
ln -sfn /content/drive/MyDrive/PTQ4SAM-Evaluation/datasets/coco data/coco

# Download train2017 to local storage to save Drive space and speed up process if missing
if [ ! -d "/content/train2017" ]; then
  echo "Downloading train2017.zip for calibration..."
  wget -q -c http://images.cocodataset.org/zips/train2017.zip -O /content/train2017.zip
  unzip -q /content/train2017.zip -d /content/
fi

# Link the train2017 to the dataset folder
ln -sfn /content/train2017 data/coco/train2017

# Download the required YOLOX-l checkpoint
rm -rf ckpt
mkdir -p ckpt
if [ ! -f "ckpt/yolox_l.pth" ]; then
  echo "Downloading YOLOX-l checkpoint..."
  wget -q -c https://download.openmmlab.com/mmdetection/v2.0/yolox/yolox_l_8x8_300e_coco/yolox_l_8x8_300e_coco_20211126_140236-d3bd2b23.pth -O ckpt/yolox_l.pth
fi

if [ ! -f "ckpt/sam_vit_b_01ec64.pth" ]; then
  echo "Copying SAM-B checkpoint..."
  cp /content/drive/MyDrive/PTQ4SAM-Evaluation/checkpoints/sam_vit_b_01ec64.pth ckpt/
fi

# Add the current directory to PYTHONPATH so local modules can be found
export PYTHONPATH=$(pwd):$PYTHONPATH

# Fix matplotlib backend issue caused by Colab's default MPLBACKEND environment variable
export MPLBACKEND=agg

$PY_BIN ptq4sam/solver/test_quant.py \
  --config ./projects/configs/yolox/yolo_l-sam-vit-b.py \
  --q_config exp/config66.yaml \
  --quant-encoder

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set the primary working directory
work_dir = '/content/drive/MyDrive/PTQ4SAM-Evaluation'
os.makedirs(work_dir, exist_ok=True)
%cd {work_dir}
print(f"Working directory set to: {work_dir}")

Mounted at /content/drive
/content/drive/MyDrive/PTQ4SAM-Evaluation
Working directory set to: /content/drive/MyDrive/PTQ4SAM-Evaluation


In [ ]:
%%bash
# Clone the PTQ4SAM repository if it doesn't exist
if [ ! -d "PTQ4SAM" ]; then
  git clone https://github.com/chengtao-lv/PTQ4SAM.git
fi

cd PTQ4SAM
# Upgrade pip and basic tools
pip install -q --upgrade pip wheel setuptools cmake

# We will bypass requirements.txt completely as it may contain hardcoded old versions
# that cause source build failures on Python 3.10+

# Install other dependencies directly
pip install -q pycocotools opencv-python onnxruntime

# Install onnx separately with strictly pre-compiled binaries
pip install -q --only-binary=:all: onnx

# Install segment_anything
pip install -q git+https://github.com/facebookresearch/segment-anything.git

echo "Dependencies installed successfully!"

Dependencies installed successfully!


In [ ]:
%%bash
# Download COCO 2017 validation data (typically used for instance segmentation evaluation)
dataset_dir="/content/drive/MyDrive/PTQ4SAM-Evaluation/datasets/coco"
mkdir -p $dataset_dir
cd $dataset_dir

echo "Downloading COCO 2017 Val images..."
if [ ! -f "val2017.zip" ]; then
  wget -q -c http://images.cocodataset.org/zips/val2017.zip
  unzip -q val2017.zip
else
  echo "val2017.zip already exists."
fi

echo "Downloading COCO 2017 annotations..."
if [ ! -f "annotations_trainval2017.zip" ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip
else
  echo "annotations_trainval2017.zip already exists."
fi

echo "COCO Dataset preparation complete!"

COCO Dataset preparation complete!


In [ ]:
%%bash
# Download SAM checkpoints into the Drive to persist across sessions
checkpoints_dir="/content/drive/MyDrive/PTQ4SAM-Evaluation/checkpoints"
mkdir -p $checkpoints_dir
cd $checkpoints_dir

echo "Downloading SAM checkpoints..."
wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth
wget -nc -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

echo "Checkpoints downloaded successfully!"

Checkpoints downloaded successfully!
